# Défi 2 NLP (medRxiv 2026) — Projet 3 : Recommandation d'articles
**Approche :** Deep Learning (Recherche Hybride Dense/BM25 & Re-ranking Cross-Encoder)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook déploie une architecture de recherche en deux étapes (*Retrieve & Re-rank*) :
1. **Recherche hybride :** combinaison des embeddings denses générés par `all-MiniLM-L6-v2` et du score lexical `BM25` ;
2. **Ré-ordonnancement neuronal :** passage des 35 meilleurs candidats dans un `Cross-Encoder` (`cross-encoder/ms-marco-MiniLM-L-6-v2`) pour affiner le classement final (**nDCG@10** et **MRR**).

## 1. Importation des bibliothèques et chargement des modèles neuronaux
Configuration de l'environnement PyTorch/Hugging Face, détection du GPU (`cuda`) et initialisation du Bi-Encodeur (`all-MiniLM-L6-v2`) ainsi que du Cross-Encoder (`ms-marco-MiniLM-L-6-v2`).

In [1]:
import os
import re
import html

os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer, CrossEncoder, util


def trouver_fichier(noms_possibles):
    for nom in noms_possibles:
        if os.path.exists(nom):
            return nom
    raise FileNotFoundError(f"Fichier introuvable : {noms_possibles}")


CORPUS_FILE = trouver_fichier([
    "medrxiv2026_corpus.csv",
    "medrxiv2026_corpus.csv.gz",
    "medrxiv2026 corpus.csv",
])
QUERIES_TRAIN_FILE = trouver_fichier([
    "medrxiv2026_reco_queries_train.csv",
    "medrxiv2026_reco_queries_train.csv.gz",
    "medrxiv2026 reco queries train.csv",
])
QRELS_TRAIN_FILE = trouver_fichier([
    "medrxiv2026_reco_qrels_train.csv",
    "medrxiv2026_reco_qrels_train.csv.gz",
    "medrxiv2026 reco qrels train.csv",
])

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Périphérique : {device}")

BI_MODEL_NAME = "all-MiniLM-L6-v2"
CROSS_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

bi_model = SentenceTransformer(BI_MODEL_NAME, device=device)
cross_model = CrossEncoder(CROSS_MODEL_NAME, max_length=512, device=device)

corpus_df = pd.read_csv(CORPUS_FILE)
queries_df = pd.read_csv(QUERIES_TRAIN_FILE)
qrels_df = pd.read_csv(QRELS_TRAIN_FILE)

Périphérique : cpu


## 2. Nettoyage et troncature *Head + Tail* des abstracts
Nettoyage des balises HTML résiduelles et troncature intelligente conservant le titre, les 150 premiers mots (contexte/méthodes) et les 80 derniers mots (résultats/conclusions) afin de respecter la fenêtre maximale du Bi-Encodeur.

In [2]:
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte).lower()
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    texte = re.sub(r"[^a-z0-9\s]", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()


# Conservation du début (introduction) et de la fin (conclusion) des résumés longs
def tronquer_head_tail(titre, abstract, max_words=230):
    mots = abstract.split()
    if len(mots) > max_words:
        abstract = " ".join(mots[:150]) + " " + " ".join(mots[-80:])
    return f"{titre}. {abstract}"


corpus_df["clean_title"] = corpus_df["title"].apply(nettoyer_texte)
corpus_df["clean_abstract"] = corpus_df["abstract"].apply(nettoyer_texte)
corpus_df["full_text"] = [
    tronquer_head_tail(t, a)
    for t, a in zip(corpus_df["clean_title"], corpus_df["clean_abstract"])
]
queries_df["clean_query"] = queries_df["query"].apply(nettoyer_texte)
doc_ids = corpus_df["ID"].values

## 3. Métriques d'évaluation et découpage Réglage / Validation
Définition des fonctions **nDCG@10** et **MRR**, et application d'un découpage 80 % réglage / 20 % validation identique au Script 1 pour comparer rigoureusement les performances.

In [3]:
TOP_K = 10


def dcg_at_k(relevances, k=10):
    relevances = np.asarray(relevances, dtype=float)[:k]
    return (
        np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))
        if relevances.size
        else 0.0
    )


def ndcg_at_k(relevances, relevances_ideales, k=10):
    idcg = dcg_at_k(sorted(relevances_ideales, reverse=True), k)
    return dcg_at_k(relevances, k) / idcg if idcg else 0.0


def rr_at_k(relevances, k=10):
    for i, r in enumerate(relevances[:k]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0


qrels_dict = {}
for q_id, d_id, rel in qrels_df[
    ["query_id", "ID", "relevance"]
].itertuples(index=False):
    qrels_dict.setdefault(q_id, {})[d_id] = rel


def evaluer_matrice(S, positions):
    nd, rr = [], []
    for i in positions:
        q_id = queries_df.loc[i, "query_id"]
        top = np.argsort(-S[i], kind="stable")[:TOP_K]
        rels = [qrels_dict[q_id].get(d, 0) for d in doc_ids[top]]
        nd.append(ndcg_at_k(rels, list(qrels_dict[q_id].values()), TOP_K))
        rr.append(rr_at_k(rels, TOP_K))
    return {
        "nDCG@10": round(float(np.mean(nd)), 4),
        "MRR": round(float(np.mean(rr)), 4),
    }


rng = np.random.RandomState(42)
perm = rng.permutation(len(queries_df))
n_dev = int(0.2 * len(perm))
DEV, TRAIN = np.sort(perm[:n_dev]), np.sort(perm[n_dev:])

## 4. Encodage sémantique dense et recherche hybride (Dense + BM25)
Calcul des embeddings vectoriels des 5 058 articles et des requêtes via `all-MiniLM-L6-v2`, puis fusion linéaire avec le score lexical `BM25` calibrée sur le jeu de réglage.

corpus_embeddings = bi_model.encode(
    corpus_df["full_text"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64,
)
query_embeddings = bi_model.encode(
    queries_df["query"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=False,
    batch_size=64,
)
S_dense = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

print(
    "Bi-Encodeur seul | Réglage :",
    evaluer_matrice(S_dense, TRAIN),
    "| Validation :",
    evaluer_matrice(S_dense, DEV),
)

# Calcul des scores BM25 normalisés
cv = CountVectorizer(stop_words="english")
tf = cv.fit_transform(corpus_df["full_text"]).tocsr().astype(np.float32)
N_docs = tf.shape[0]
df_counts = np.bincount(tf.indices, minlength=tf.shape[1])
idf = np.log(1 + (N_docs - df_counts + 0.5) / (df_counts + 0.5)).astype(
    np.float32
)
tf_coo = tf.tocoo()
dl = np.asarray(tf.sum(1)).ravel()
avgdl = dl.mean()
k1, b = 1.2, 0.9
denom = tf_coo.data + k1 * (1 - b + b * dl[tf_coo.row] / avgdl)
W_bm25 = sp.csr_matrix(
    (
        tf_coo.data * (k1 + 1) / denom * idf[tf_coo.col],
        (tf_coo.row, tf_coo.col),
    ),
    shape=tf.shape,
)


def calculer_bm25_norm(clean_queries):
    Q = cv.transform(clean_queries).tocsr()
    Q.data[:] = 1.0
    S = (Q @ W_bm25.T).toarray()
    return S / np.maximum(S.max(axis=1, keepdims=True), 1e-9)


S_bm25_norm = calculer_bm25_norm(queries_df["clean_query"])

# Calibrage du coefficient hybride alpha sur TRAIN
best_alpha = 0.85
best_hyb_score = -1
for alpha in [0.75, 0.80, 0.85, 0.90, 0.95]:
    sc = evaluer_matrice(alpha * S_dense + (1 - alpha) * S_bm25_norm, TRAIN)[
        "nDCG@10"
    ]
    if sc > best_hyb_score:
        best_hyb_score, best_alpha = sc, alpha

S_hybrid = best_alpha * S_dense + (1 - best_alpha) * S_bm25_norm
print(
    f"Hybride (alpha={best_alpha}) | Réglage :",
    evaluer_matrice(S_hybrid, TRAIN),
    "| Validation :",
    evaluer_matrice(S_hybrid, DEV),
)

In [4]:
corpus_embeddings = bi_model.encode(
    corpus_df["full_text"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64,
)
query_embeddings = bi_model.encode(
    queries_df["query"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=False,
    batch_size=64,
)
S_dense = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

print(
    "Bi-Encodeur seul | Réglage :",
    evaluer_matrice(S_dense, TRAIN),
    "| Validation :",
    evaluer_matrice(S_dense, DEV),
)

# Calcul des scores BM25 normalisés
cv = CountVectorizer(stop_words="english")
tf = cv.fit_transform(corpus_df["full_text"]).tocsr().astype(np.float32)
N_docs = tf.shape[0]
df_counts = np.bincount(tf.indices, minlength=tf.shape[1])
idf = np.log(1 + (N_docs - df_counts + 0.5) / (df_counts + 0.5)).astype(
    np.float32
)
tf_coo = tf.tocoo()
dl = np.asarray(tf.sum(1)).ravel()
avgdl = dl.mean()
k1, b = 1.2, 0.9
denom = tf_coo.data + k1 * (1 - b + b * dl[tf_coo.row] / avgdl)
W_bm25 = sp.csr_matrix(
    (
        tf_coo.data * (k1 + 1) / denom * idf[tf_coo.col],
        (tf_coo.row, tf_coo.col),
    ),
    shape=tf.shape,
)


def calculer_bm25_norm(clean_queries):
    Q = cv.transform(clean_queries).tocsr()
    Q.data[:] = 1.0
    S = (Q @ W_bm25.T).toarray()
    return S / np.maximum(S.max(axis=1, keepdims=True), 1e-9)


S_bm25_norm = calculer_bm25_norm(queries_df["clean_query"])

# Calibrage du coefficient hybride alpha sur TRAIN
best_alpha = 0.85
best_hyb_score = -1
for alpha in [0.75, 0.80, 0.85, 0.90, 0.95]:
    sc = evaluer_matrice(alpha * S_dense + (1 - alpha) * S_bm25_norm, TRAIN)[
        "nDCG@10"
    ]
    if sc > best_hyb_score:
        best_hyb_score, best_alpha = sc, alpha

S_hybrid = best_alpha * S_dense + (1 - best_alpha) * S_bm25_norm
print(
    f"Hybride (alpha={best_alpha}) | Réglage :",
    evaluer_matrice(S_hybrid, TRAIN),
    "| Validation :",
    evaluer_matrice(S_hybrid, DEV),
)

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Bi-Encodeur seul | Réglage : {'nDCG@10': 0.7276, 'MRR': 0.9183} | Validation : {'nDCG@10': 0.7447, 'MRR': 0.9468}
Hybride (alpha=0.9) | Réglage : {'nDCG@10': 0.7502, 'MRR': 0.9338} | Validation : {'nDCG@10': 0.757, 'MRR': 0.9594}


## 5. Ré-ordonnancement neuronal (*Re-ranking*) par Cross-Encoder
Extraction des 35 meilleurs candidats par requête issus du score hybride et évaluation conjointe de chaque paire `(requête, article)` par le `Cross-Encoder` afin d'affiner l'ordre du Top-10.

In [5]:
TOP_CANDIDATES = 35
full_texts_raw = (
    corpus_df["title"].fillna("") + ". " + corpus_df["abstract"].fillna("")
).tolist()
queries_raw = queries_df["query"].tolist()

S_rerank = np.copy(S_hybrid)
all_pairs = []
top_indices_all = np.zeros((len(queries_df), TOP_CANDIDATES), dtype=int)

for i in range(len(queries_df)):
    top_c = np.argpartition(-S_hybrid[i], TOP_CANDIDATES - 1)[:TOP_CANDIDATES]
    top_c = top_c[np.argsort(-S_hybrid[i, top_c])]
    top_indices_all[i] = top_c
    for doc_idx in top_c:
        all_pairs.append([queries_raw[i], full_texts_raw[doc_idx]])

ce_logits = cross_model.predict(
    all_pairs, batch_size=64, show_progress_bar=True
)
ce_probs = (1.0 / (1.0 + np.exp(-np.asarray(ce_logits)))).reshape(
    len(queries_df), TOP_CANDIDATES
)

# Calibrage du poids du Cross-Encoder (beta) sur TRAIN
best_beta = 0.5
best_rerank_score = -1
for beta in [0.0, 0.25, 0.40, 0.50, 0.65, 0.80]:
    S_temp = np.copy(S_hybrid)
    for i in range(len(queries_df)):
        idx_c = top_indices_all[i]
        S_temp[i, idx_c] = (
            10.0 + beta * ce_probs[i] + (1.0 - beta) * S_hybrid[i, idx_c]
        )
    sc = evaluer_matrice(S_temp, TRAIN)["nDCG@10"]
    if sc > best_rerank_score:
        best_rerank_score, best_beta = sc, beta

for i in range(len(queries_df)):
    idx_c = top_indices_all[i]
    S_rerank[i, idx_c] = (
        10.0 + best_beta * ce_probs[i] + (1.0 - best_beta) * S_hybrid[i, idx_c]
    )

print(
    f"Hybride + Cross-Encoder (beta={best_beta}) | Réglage :",
    evaluer_matrice(S_rerank, TRAIN),
    "| Validation :",
    evaluer_matrice(S_rerank, DEV),
)

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Hybride + Cross-Encoder (beta=0.25) | Réglage : {'nDCG@10': 0.7622, 'MRR': 0.9476} | Validation : {'nDCG@10': 0.7649, 'MRR': 0.9532}


## 6. Inférence sur le jeu de test et génération du fichier de soumission
Application de la chaîne complète (Bi-Encodeur + BM25 + Cross-Encoder Re-ranking) sur les requêtes de test et export du fichier final `Défi2_DL_Projet3`.

In [6]:
def generer_predictions_test(
    fichier_test_queries, fichier_sortie="Défi2_DL_Projet3.csv"
):
    test_q_df = pd.read_csv(fichier_test_queries)
    test_q_raw = test_q_df["query"].fillna("").tolist()
    test_q_clean = test_q_df["query"].apply(nettoyer_texte)

    test_q_emb = bi_model.encode(
        test_q_raw, convert_to_tensor=True, show_progress_bar=False, batch_size=64
    )
    s_dense_test = util.cos_sim(test_q_emb, corpus_embeddings).cpu().numpy()

    s_bm25_test = calculer_bm25_norm(test_q_clean)
    s_hyb_test = best_alpha * s_dense_test + (1.0 - best_alpha) * s_bm25_test

    # Ré-ordonnancement du Top-35 par Cross-Encoder
    test_pairs = []
    test_top_c = np.zeros((len(test_q_df), TOP_CANDIDATES), dtype=int)
    for i in range(len(test_q_df)):
        tc = np.argpartition(-s_hyb_test[i], TOP_CANDIDATES - 1)[
            :TOP_CANDIDATES
        ]
        tc = tc[np.argsort(-s_hyb_test[i, tc])]
        test_top_c[i] = tc
        for doc_idx in tc:
            test_pairs.append([test_q_raw[i], full_texts_raw[doc_idx]])

    test_ce_logits = cross_model.predict(
        test_pairs, batch_size=64, show_progress_bar=True
    )
    test_ce_probs = (1.0 / (1.0 + np.exp(-np.asarray(test_ce_logits)))).reshape(
        len(test_q_df), TOP_CANDIDATES
    )

    resultats = []
    for i, q_id in enumerate(test_q_df["query_id"]):
        idx_c = test_top_c[i]
        scores_finaux = (
            best_beta * test_ce_probs[i]
            + (1.0 - best_beta) * s_hyb_test[i, idx_c]
        )
        best_10_pos = np.argsort(-scores_finaux, kind="stable")[:TOP_K]
        for rank_pos, pos in enumerate(best_10_pos, start=1):
            doc_idx = idx_c[pos]
            resultats.append({
                "query_id": q_id,
                "ID": doc_ids[doc_idx],
                "rank": rank_pos,
            })

    df_res = pd.DataFrame(resultats)
    df_res.to_csv(fichier_sortie, index=False)
    print(f"Fichier généré : {fichier_sortie} ({df_res.shape})")


for test_candidate in [
    "medrxiv2026 reco queries test.csv",
    "medrxiv2026_reco_queries_test.csv",
    "medrxiv2026 reco queries test.csv.gz",
    "medrxiv2026_reco_queries_test.csv.gz",
]:
    if os.path.exists(test_candidate):
        generer_predictions_test(
            test_candidate, fichier_sortie="Défi2_DL_Projet3.csv"
        )
        break

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

Fichier généré : Défi2_DL_Projet3.csv ((2000, 3))
